# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faith-amanze/content-refresh-prioritization/blob/main/work/notebooks/w04_signal_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

Impressions, clicks, sessions, and staleness are all heavy-tailed (median far below mean, p95
still far below max) — a few giant pages dominate raw totals. That's why every test below reads
grouped medians/rates by bucket rather than a raw correlation on the untransformed numbers.

In [1]:
# ── Distributions first (heavy tails change how everything below should be read) ──
import pandas as pd
import numpy as np
import os

_candidates = [
    "/workspaces/assignment1/data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
    "data/raw/content_refresh_anonymized.csv",
]
_data_path = next(p for p in _candidates if os.path.exists(p))
df = pd.read_csv(_data_path)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

for col in ["impressions_90d", "clicks_90d", "sessions_90d", "days_since_last_update"]:
    d = df[col]
    print(f"{col:<24} mean={d.mean():>9.1f}  median={d.median():>7.1f}  p95={d.quantile(.95):>9.1f}  max={d.max():>10.1f}")

print("\nAll four are heavy-tailed: median << mean, and p95 is still far below max.")
print("A few giant pages dominate raw totals -- grouped medians / tiers (not raw Pearson")
print("correlation) are the safer read below, same approach as the flyrank data dictionary warns.")


impressions_90d          mean=   5200.4  median=  731.0  p95=  22996.5  max=  517715.0
clicks_90d               mean=     16.1  median=    1.0  p95=     69.0  max=    4178.0
sessions_90d             mean=     37.1  median=    7.0  p95=    166.0  max=    4345.0
days_since_last_update   mean=     46.1  median=   20.0  p95=    104.0  max=     373.0

All four are heavy-tailed: median << mean, and p95 is still far below max.
A few giant pages dominate raw totals -- grouped medians / tiers (not raw Pearson
correlation) are the safer read below, same approach as the flyrank data dictionary warns.


## 2. Signal test #1 / #2 / #3 (verdict each)

Three signals relevant to the Content Refresh Prioritization lane, each with a mini-test and a
verdict, and n's shown so a thin bucket can't masquerade as a real pattern.

In [2]:
# ── Signal test 1: CTR vs position (does a better rank actually earn more clicks?) ──
df_pos = df[df["avg_position"] > 0].copy()
df_pos["position_bucket"] = pd.cut(df_pos["avg_position"], bins=[0,3,10,20,50,1000],
                                    labels=["1-3","4-10","11-20","21-50","50+"])
s1 = df_pos.groupby("position_bucket", observed=True).agg(
    n=("ctr","size"), avg_ctr=("ctr","mean")).reset_index()
print("Signal 1 -- CTR by position bucket")
print(s1)
print("VERDICT: CONFIRMED. CTR drops monotonically as position worsens (2.71% at 1-3 down to")
print("~0.15% at 50+), across buckets from n=1,141 to n=11,842 -- well above the sample-size floor.\n")

# ── Signal test 2: content_type vs declining rate ──
s2 = df.groupby("content_type").agg(n=("is_declining_label","size"),
                                     decline_rate=("is_declining_label","mean")).reset_index()
print("Signal 2 -- decline rate by content_type")
print(s2)
print("VERDICT: CONFIRMED. Feedly articles decline far less often (28.7%, n=2,096) than keyword")
print("(56.1%, n=27,207) or comparison articles (57.2%, n=697) -- all buckets comfortably above")
print("the sample-size floor, so this is a real, usable split, not noise.\n")

# ── Signal test 3: staleness vs declining rate ──
df["staleness_bucket"] = pd.cut(df["days_since_last_update"], bins=[-1,30,90,180,365,99999],
                                 labels=["0-30","31-90","91-180","181-365","365+"])
s3 = df.groupby("staleness_bucket", observed=True).agg(
    n=("is_declining_label","size"), decline_rate=("is_declining_label","mean")).reset_index()
print("Signal 3 -- decline rate by staleness bucket")
print(s3)
print("VERDICT: MIXED. The two large buckets (0-30, n=20,480 and 91-180, n=9,171) show a real")
print("gap -- 51.1% vs 61.1% decline rate -- so staler pages DO decline somewhat more. But 31-90")
print("(n=175), 181-365 (n=169), and especially 365+ (n=5) are all below or near the sample-size")
print("floor and swing in ways that don't form a clean trend -- not enough to call the FULL")
print("staleness-decline relationship confirmed, only the large-bucket comparison.")


Signal 1 -- CTR by position bucket
  position_bucket      n   avg_ctr
0             1-3   1141  2.714303
1            4-10  11842  0.651045
2           11-20   7273  0.323443
3           21-50   7225  0.222345
4             50+   1314  0.150784
VERDICT: CONFIRMED. CTR drops monotonically as position worsens (2.71% at 1-3 down to
~0.15% at 50+), across buckets from n=1,141 to n=11,842 -- well above the sample-size floor.

Signal 2 -- decline rate by content_type
         content_type      n  decline_rate
0  comparison article    697      0.572453
1      feedly article   2096      0.286737
2     keyword article  27207      0.560959
VERDICT: CONFIRMED. Feedly articles decline far less often (28.7%, n=2,096) than keyword
(56.1%, n=27,207) or comparison articles (57.2%, n=697) -- all buckets comfortably above
the sample-size floor, so this is a real, usable split, not noise.

Signal 3 -- decline rate by staleness bucket
  staleness_bucket      n  decline_rate
0             0-30  20480      

## 3. The flag-linked test

My w05 model (and w02 framing) use `trend_direction == "down"` as the review flag. That only
makes sense if "declining" also means "currently underperforming" — otherwise the flag could
fire on a noisy recent dip while the page is actually fine. Testing that assumption directly.

In [3]:
# ── The flag-linked test ──
# The w05 model (and my w02 framing) treat "trend_direction == down" as the thing worth
# flagging for review. That only makes sense if "declining" also means "currently
# performing worse" -- otherwise a page could be flagged just for a noisy recent dip while
# actually being fine. Test: does the label track current CTR and position?
flag_test = df_pos.groupby("trend_direction", observed=True).agg(
    n=("ctr", "size"), avg_ctr=("ctr", "mean"), avg_position=("avg_position", "mean")
).reset_index().sort_values("avg_ctr")
print(flag_test)

print("""
VERDICT: MIXED. "down" pages do have the lowest average CTR (0.32%) of any trend group,
which supports using the label as a review flag -- n=16,254, well above the floor. But
"down" pages are NOT the worst on position (15.9, close to "stable" pages at 16.3) --
"up" trending pages actually have the WORST average position (22.5) while pulling a
decent CTR (0.57%). So the label captures "worse clicks lately" reasonably well, but it
is not a stand-in for "worst overall performer" -- a page can be trending up while still
ranking badly, and that combination wouldn't get flagged by this label alone.
""")


  trend_direction      n   avg_ctr  avg_position
0            down  16254  0.324297     15.944149
3          stable   5962  0.517321     16.332623
4              up   4388  0.565533     22.512739
1            flat   1109  1.339964     11.532913
2             new   1082  2.440564     20.484473

VERDICT: MIXED. "down" pages do have the lowest average CTR (0.32%) of any trend group,
which supports using the label as a review flag -- n=16,254, well above the floor. But
"down" pages are NOT the worst on position (15.9, close to "stable" pages at 16.3) --
"up" trending pages actually have the WORST average position (22.5) while pulling a
decent CTR (0.57%). So the label captures "worse clicks lately" reasonably well, but it
is not a stand-in for "worst overall performer" -- a page can be trending up while still
ranking badly, and that combination wouldn't get flagged by this label alone.



## 4. What this means in practice

In [4]:
print("Practical takeaway, in numbers a content lead can act on:")
print(f"- Position 11+ pages average {df_pos[df_pos['avg_position']>=11]['ctr'].mean():.2f}% CTR vs "
      f"{df_pos[df_pos['avg_position']<11]['ctr'].mean():.2f}% for top-10 -- the CTR/position link "
      f"is the strongest, cleanest signal in this audit and safe to build a rule on.")
print(f"- Feedly-sourced content behaves differently ({df[df['content_type']=='feedly article']['is_declining_label'].mean():.0%} "
      f"decline rate vs {df[df['content_type']=='keyword article']['is_declining_label'].mean():.0%} for keyword articles) -- "
      f"worth segmenting by content_type rather than one blanket rule.")


Practical takeaway, in numbers a content lead can act on:
- Position 11+ pages average 0.26% CTR vs 0.80% for top-10 -- the CTR/position link is the strongest, cleanest signal in this audit and safe to build a rule on.
- Feedly-sourced content behaves differently (29% decline rate vs 56% for keyword articles) -- worth segmenting by content_type rather than one blanket rule.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
